# Notebook 6: Backtesting & Calibration Loop

**Goal:** Simulate what would have happened if you'd used YakOS projections on historical slates.

**Input:** `yakos_features.parquet`, trained models from Notebooks 3–5

**Output:** Calibration report, updated config adjustments

## Steps
For each historical slate date:
1. Generate YakOS projections (FP, minutes, ownership) using the models
2. Run the optimizer with those projections
3. Score lineups against actual results
4. Compare against Tank01 raw, RG raw, and YakOS ensemble
5. Track per-slate MAE, bias, top-lineup score, cash rate
6. Identify systematic patterns for calibration

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import SimpleImputer

INPUT_PATH = 'yakos_features.parquet'
MODEL_DIR  = Path('models')

df = pd.read_parquet(INPUT_PATH)
df['game_date'] = pd.to_datetime(df['game_date'])
df = df.dropna(subset=['dk_fp_actual'])  # Only dates with known actuals

print(f'Backtest rows: {len(df)}')
print(f'Dates: {df["game_date"].nunique()}')

## Load Trained Models

In [ ]:
def load_model_and_features(model_name):
    """Load a trained model and its feature list."""
    model_path   = MODEL_DIR / f'{model_name}.pkl'
    feature_path = MODEL_DIR / f'{model_name.replace("model", "features")}.json'
    if not model_path.exists():
        return None, []
    model    = joblib.load(model_path)
    features = json.load(open(feature_path)) if feature_path.exists() else []
    return model, features


fp_model,    fp_features    = load_model_and_features('yakos_fp_model')
min_model,   min_features   = load_model_and_features('yakos_minutes_model')
own_model,   own_features   = load_model_and_features('yakos_ownership_model')

print(f'FP model:        {"loaded" if fp_model else "not found"}')
print(f'Minutes model:   {"loaded" if min_model else "not found"}')
print(f'Ownership model: {"loaded" if own_model else "not found"}')

## Generate YakOS Projections for All Historical Rows

In [ ]:
def predict_with_fallback(model, features, df, fallback_col):
    """Predict using model if available; fall back to a named column."""
    if model is None or not features:
        return df.get(fallback_col, pd.Series(np.nan, index=df.index))
    available = [f for f in features if f in df.columns]
    if not available:
        return df.get(fallback_col, pd.Series(np.nan, index=df.index))
    X = df[available]
    imp = SimpleImputer(strategy='median')
    X_imp = imp.fit_transform(X)
    return pd.Series(model.predict(X_imp), index=df.index)


df['yakos_fp_pred']  = predict_with_fallback(fp_model,  fp_features,  df, 'rolling_fp_10')
df['yakos_min_pred'] = predict_with_fallback(min_model, min_features, df, 'rolling_min_10')
df['yakos_own_pred'] = predict_with_fallback(own_model, own_features, df, 'rg_ownership')

# Ensemble: 40% YakOS + 30% Tank01 + 30% RG
def safe_series(df, col):
    return pd.to_numeric(df.get(col, pd.Series(np.nan, index=df.index)), errors='coerce')

yakos_preds  = safe_series(df, 'yakos_fp_pred')
tank01_preds = safe_series(df, 'tank01_proj')
rg_preds     = safe_series(df, 'rg_proj')

# Weighted ensemble with NaN redistribution
w = {'yakos': 0.40, 'tank01': 0.30, 'rg': 0.30}
ensemble = pd.Series(np.nan, index=df.index, dtype=float)
for i in df.index:
    vals, weights = [], []
    for v, wk in [(yakos_preds.get(i), w['yakos']),
                  (tank01_preds.get(i), w['tank01']),
                  (rg_preds.get(i), w['rg'])]:
        if v is not None and not np.isnan(float(v)):
            vals.append(float(v))
            weights.append(wk)
    tw = sum(weights)
    ensemble[i] = sum(v*wk for v,wk in zip(vals,weights)) / tw if tw > 0 else np.nan

df['ensemble_proj'] = ensemble
print('Projections generated.')

## Per-Slate Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error

def evaluate_proj(actual, predicted, name):
    """Compute MAE, bias, hit rate for a projection series."""
    mask = predicted.notna() & actual.notna()
    if mask.sum() == 0:
        return {'name': name, 'n': 0, 'mae': None, 'bias': None, 'hit_rate_10': None}
    a, p = actual[mask], predicted[mask]
    err = p - a
    return {
        'name': name,
        'n': int(mask.sum()),
        'mae': round(float(mean_absolute_error(a, p)), 2),
        'bias': round(float(err.mean()), 2),
        'hit_rate_10': round(float((err.abs() <= 10).mean() * 100), 1),
    }


actuals = df['dk_fp_actual']

print('=== Overall Projection Quality ===')
for proj_col, label in [
    ('tank01_proj',   'Tank01 Raw'),
    ('rg_proj',       'RotoGrinders Raw'),
    ('yakos_fp_pred', 'YakOS Model'),
    ('ensemble_proj', 'YakOS Ensemble (40/30/30)'),
]:
    if proj_col in df.columns:
        stats = evaluate_proj(actuals, df[proj_col], label)
        print(f"  {label:35s}: MAE={stats.get('mae', 'N/A'):>6} | Bias={stats.get('bias', 'N/A'):>+6} | Hit%={stats.get('hit_rate_10', 'N/A'):>5}")

## Per-Date Calibration Report

In [ ]:
slate_reports = []
for date, grp in df.groupby('game_date'):
    row = {'game_date': date, 'n_players': len(grp)}
    for proj_col, label in [('tank01_proj', 'tank01'), ('yakos_fp_pred', 'yakos'), ('ensemble_proj', 'ensemble')]:
        if proj_col in grp.columns:
            stats = evaluate_proj(grp['dk_fp_actual'], grp[proj_col], label)
            row[f'{label}_mae']  = stats.get('mae')
            row[f'{label}_bias'] = stats.get('bias')
    slate_reports.append(row)

slate_df = pd.DataFrame(slate_reports).sort_values('game_date')
print('Per-slate calibration report:')
print(slate_df.to_string())

## Systematic Pattern Analysis

In [ ]:
if 'ensemble_proj' in df.columns and df['ensemble_proj'].notna().any():
    df['proj_error'] = df['ensemble_proj'] - df['dk_fp_actual']

    # Error by position
    if 'pos' in df.columns:
        pos_err = df.groupby('pos')['proj_error'].agg(['mean', 'std', 'count']).round(2)
        pos_err.columns = ['bias', 'std', 'count']
        print('\nError by position (positive bias = over-projecting):')
        print(pos_err)

    # Error by salary bracket
    if 'salary' in df.columns:
        df['salary_bracket'] = pd.cut(df['salary'], bins=[0, 5000, 6000, 7000, 8000, 100000],
                                       labels=['<5K', '5-6K', '6-7K', '7-8K', '>8K'])
        sal_err = df.groupby('salary_bracket', observed=True)['proj_error'].agg(['mean', 'count']).round(2)
        sal_err.columns = ['bias', 'count']
        print('\nError by salary bracket:')
        print(sal_err)

    # Back-to-back bias
    if 'b2b' in df.columns:
        b2b_err = df.groupby('b2b')['proj_error'].mean().round(2)
        print(f'\nB2B bias: normal={b2b_err.get(0, "N/A"):+.2f}, B2B={b2b_err.get(1, "N/A"):+.2f}')

## Save Calibration Config Adjustments

In [ ]:
# Compute suggested calibration adjustments based on per-position bias
calibration_notes = []

if 'pos' in df.columns and 'proj_error' in df.columns:
    pos_bias = df.groupby('pos')['proj_error'].mean().to_dict()
    for pos, bias in pos_bias.items():
        if abs(bias) > 2.0:
            direction = 'over-projecting' if bias > 0 else 'under-projecting'
            calibration_notes.append(
                f'{pos}: {direction} by {abs(bias):.1f} FP — '
                f'adjust DvP weighting or add position-level bias correction'
            )

if calibration_notes:
    print('\n=== Calibration Recommendations ===')
    for note in calibration_notes:
        print(f'  • {note}')
else:
    print('No significant systematic biases detected.')

# Save report
report = {
    'overall_ensemble_mae': slate_df.get('ensemble_mae', pd.Series()).mean() if 'ensemble_mae' in slate_df.columns else None,
    'slate_count': len(slate_df),
    'calibration_notes': calibration_notes,
}
with open('yakos_calibration_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)
print('\nCalibration report saved: yakos_calibration_report.json')